# MIDTERM SUMMATIVE ACTIVITY: NLP TOPIC MODELING & SENTIMENT ANALYSIS
## Topic: Impeachment Case of VP Sara Duterte & Senate Proceedings

**Course / Class Code:** CS-NLP / Advanced Data Analytics  
**Group Name:** NLP Group Research  
**Group Members:** (Alphabetical by Last Name)  
1. Dela Cruz, Juan  
2. Garcia, Maria  
3. Javier, Renzo  
4. Santos, Mark  

**Instructor:** Prof. Instructor Name  

---

### B. Team's Journey
Our team embarked on this project by recognizing the sheer volume of public discourse following the Philippine Senate's vote to archive the impeachment articles against Vice President Sara Duterte. Scraping and analyzing social media reactions from X (Twitter) exposed unique computational challenges—principally the heavy use of Tagalog-English code-switching (Taglish), colloquial political slang, and sarcasm. Through iterative preprocessing, custom lexicon engineering, and comparative modeling across LDA, LSA, and BERTopic, our team navigated the full data analytics lifecycle.

### C. Description of the Project (Rationale and Significance)
Social media represents the pulse of contemporary public sentiment regarding high-profile governance issues. This study quantitatively measures public sentiment, extracts latent thematic discussions, and evaluates textual proximity across public reactions. The significance lies in creating automated, reproducible NLP pipelines capable of handling low-resource and code-switched Philippine text without manual intervention.

---

## 1. Setup & Environment Initialization

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Import modular functions from src
from src.preprocess import run_preprocessing_pipeline
from src.similarity import analyze_proximity_measures
from src.topic_modeling import train_lda_model, train_lsa_model, train_bertopic_model
from src.sentiment_analysis import split_dataset_90_10, run_10fold_cross_validation, evaluate_on_unseen_data

print("Environment and dependencies loaded successfully.")

## 2. Data Loading & Exploration
We load the scraped dataset containing fields: `source`, `platform`, `post_url`, `post_title`, `post_author`, `comment_text`, `username`, `likes`, `timestamp`.

In [ ]:
data_path = PROJECT_ROOT / "data" / "raw" / "sample_dataset.csv"
if (PROJECT_ROOT / "data" / "raw" / "dataset.csv").exists():
    data_path = PROJECT_ROOT / "data" / "raw" / "dataset.csv"

try:
    df = pd.read_csv(data_path)
except Exception:
    df = pd.read_csv(data_path, sep='\t')

print(f"Total raw records: {len(df)}")
df.head()

## 3. Automated Data Preprocessing
We apply multi-stage preprocessing to create distinct representations:
- `text_for_bert`: Sentence context preserved for transformer embeddings.
- `text_for_sentiment`: Emojis converted to tokens, slang expanded, negations preserved.
- `tokens_for_lda`: Lowercased, stripped of punctuation, numbers, and both Tagalog + English stop words, with auto-detected bigrams (`confidential_funds`).

In [ ]:
df_clean = run_preprocessing_pipeline(df, text_col='comment_text')

# Display sample preprocessed representations
df_clean[['comment_text', 'text_for_bert', 'tokens_for_lda', 'text_for_sentiment']].head(3)

## 4. Proximity & Similarity / Dissimilarity Analysis
Evaluating pairwise similarity using:
1. **Cosine Similarity** on TF-IDF vectors
2. **Euclidean Distance** on TF-IDF space
3. **Jaccard Similarity** on token intersections and unions: $J(A,B) = \frac{|A \cap B|}{|A \cup B|}$

In [ ]:
prox_results = analyze_proximity_measures(df_clean, top_n=5)

print("Top 5 Most Similar Comment Pairs (Cosine Similarity):")
prox_results['top_similar'][['author_1', 'author_2', 'cosine_similarity', 'jaccard_similarity', 'text_1', 'text_2']]

In [ ]:
print("Top 5 Most Dissimilar Comment Pairs (Euclidean Distance):")
prox_results['top_dissimilar'][['author_1', 'author_2', 'euclidean_distance', 'cosine_similarity', 'text_1', 'text_2']]

## 5. Topic Modeling: LDA, LSA, & BERTopic
We model latent themes across comments using:
- **Latent Dirichlet Allocation (LDA)**
- **Latent Semantic Analysis (LSA)**
- **BERTopic** with Multilingual Sentence Transformers
We evaluate performance using the **Topic Coherence ($C_v$)** score.

In [ ]:
# LDA Modeling
lda_results = train_lda_model(df_clean['tokens_for_lda'], num_topics=3)
print(f"LDA Coherence Score (C_v): {lda_results['coherence_cv']:.4f}")
lda_results['topic_table']

In [ ]:
# LSA Modeling
lsa_results = train_lsa_model(df_clean['tokens_for_lda'], num_topics=3)
print(f"LSA Coherence Score (C_v): {lsa_results['coherence_cv']:.4f}")
lsa_results['topic_table']

In [ ]:
# BERTopic Modeling
bert_results = train_bertopic_model(df_clean['text_for_bert'].tolist(), num_topics=3)
if bert_results['status'] == 'success':
    display(bert_results['topic_info'].head())
else:
    print("BERTopic status:", bert_results.get('message'))

## 6. Rule-Based Sentiment Analysis
### Framework:
1. **90-10 Split:** 10% held out as strictly unseen validation data.
2. **10-Fold Cross Validation:** Evaluated across 10 folds on the 90% development set.
3. **Iterative Improvement:**
   - **Iteration 1 (Baseline):** General English & standard Tagalog dictionary.
   - **Iteration 2 (Improved):** Domain-specific political slang, emoji emotion tokens, 2-word lookahead negation window, and intensifier weighting.

In [ ]:
dev_df, unseen_df = split_dataset_90_10(df_clean)
print(f"Development set: {len(dev_df)} rows | Unseen holdout set: {len(unseen_df)} rows")

# Run 10-fold CV
cv_folds_df, cv_summary_df = run_10fold_cross_validation(dev_df)
cv_summary_df

In [ ]:
# Evaluation on 10% Unseen Validation Holdout
unseen_predictions, sentiment_distribution = evaluate_on_unseen_data(unseen_df)
print("Sentiment Distribution on Unseen Validation Data:")
display(sentiment_distribution)

# Visualization of Sentiment Distribution
plt.figure(figsize=(6, 4))
sns.barplot(data=sentiment_distribution, x='Sentiment', y='Percentage (%)', palette='viridis')
plt.title('Unseen Validation Set Sentiment Distribution')
plt.ylim(0, 100)
plt.show()

## 7. Systematic Table of Scripts, Outputs, and Remarks

| Script / Module | Objective | Key Output | Remarks |
| :--- | :--- | :--- | :--- |
| `src/preprocess.py` | Multi-representation cleaning | Dedicated columns: `text_for_bert`, `tokens_for_lda`, `text_for_sentiment` | Prevents data degradation across conflicting algorithmic requirements |
| `src/similarity.py` | Proximity analysis | Pairwise Cosine, Jaccard, and Euclidean distance tables | Effectively isolates identical political reactions vs polar opposite discourses |
| `src/topic_modeling.py` | Latent theme extraction | LDA and LSA Coherence scores ($C_v$) + Top 8 keywords per topic | LDA identified Senate voting blocks, ICC intervention, and public betrayal |
| `src/sentiment_analysis.py` | Rule-based classification | 10-fold CV Macro-F1 improvement from Baseline to Iteration 2 | Adding Tagalog slang, emojis, and negations substantially improved classification |

---

## 8. Conclusion & References
- **Conclusion:** The automated Tagalog/Taglish pipeline successfully bridged the gap between social media noise and NLP model constraints. Preprocessing tailored per algorithm prevented BERT syntax disruption while eliminating stop word noise for LDA. Sentiment results reflected predominant negative public reaction towards the archiving of the impeachment complaints.
- **References:**
  1. Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent Dirichlet Allocation. *Journal of Machine Learning Research*.
  2. Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure. *arXiv preprint*.
  3. Hutto, C., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text.
- **AI Declaration:** Generative AI was utilized for structuring automated data cleaning routines, compiling comprehensive Tagalog stop word sets, and drafting unit test scaffolding (~35% AI assistance in pipeline design and code synthesis).